### Phase 5: Evaluation & Vergleich (Der Kern der Thesis)
Beweis des Mehrwerts zur Reduktion des **Sequence of Returns Risk**.
*   **Metriken:**
    *   **Maximum Drawdown (MDD):** Wie tief war der maximale Fall? (Wichtigstes Ziel).
    *   **Sharpe Ratio / Sortino Ratio:** Risiko-adjustierte Rendite.
    *   **Calmar Ratio:** Rendite im Verhältnis zum Drawdown.
    *   **Regime-Stabilität:** Wie oft schaltet das Modell um? (LSTMs neigen zum "Overfitting" und Rauschen).

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))
backtesting_transaction_costs = pd.read_parquet(cfg.data_path("backtesting_costs"))
test_df = pd.read_parquet(cfg.data_path("test_data"))

In [ ]:
import numpy as np

# 1. Daten aus dem data-Ordner laden (bereits in Zelle 1 geladen, aber hier nochmal sichergestellt)
test_df = pd.read_parquet(cfg.data_path("test_data"))
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))

# 2. Funktion zur Evaluation
def evaluate_strategies(results_df, trades_df, costs_df):
    stats = []
    
    for col in results_df.columns:
        equity_curve = results_df[col]
        daily_returns = equity_curve.pct_change().dropna()
        
        # 1. Total Return & CAGR (Annualisierte Rendite)
        total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0]) - 1
        days = (equity_curve.index[-1] - equity_curve.index[0]).days
        cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0])**(365.25/days) - 1
        
        # 2. Volatilität (annualisiert)
        vol = daily_returns.std() * np.sqrt(252)
        
        # 3. Sharpe Ratio (Annahme: Risk-Free Rate = 0, da Cash bereits in der Strategie (im Portfolio) steckt)
        sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(252) if daily_returns.std() != 0 else 0
        
        # 4. Maximum Drawdown
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve / peak) - 1
        mdd = drawdown.min()
        
        # 5. Sortino Ratio (Fokus auf Downside-Risiko)
        downside_returns = daily_returns[daily_returns < 0]
        downside_std = downside_returns.std() * np.sqrt(252)
        sortino = (daily_returns.mean() * 252) / downside_std if downside_std != 0 else np.nan
        
        # 6. Calmar Ratio (Verhältnis Rendite zu Max Drawdown)
        calmar = cagr / abs(mdd) if mdd != 0 else np.nan
        
        # 7. Anzahl der Trades (Regime-Wechsel)
        if col in trades_df.columns:
            switches = trades_df[col].diff().abs().sum()
        else:
            switches = 0
        
        # 8. Gesamte Transaktionskosten am Ende des Zeitraums extrahieren
        if col in costs_df.columns:
            total_fees = costs_df[col].iloc[-1]
        else:
            total_fees = 0.0
            
        stats.append({
            'Strategie': col.replace('_', ' '),
            'Total Return': f"{total_return:.2%}",
            'CAGR (p.a.)': f"{cagr:.2%}",
            'Volatilität': f"{vol:.2%}",
            'Max Drawdown': f"{mdd:.2%}",
            'Sharpe Ratio': round(sharpe, 2),
            'Sortino Ratio': round(sortino, 2),
            'Calmar Ratio': round(calmar, 2),
            'Regime-Wechsel': int(switches),
            'Gesamtkosten (Gebühren)': f"{total_fees:.2%}"
        })
    
    return pd.DataFrame(stats).set_index('Strategie')

# --- 3. DYNAMISCHE ZUORDNUNG DER SIGNALE ---
signals_to_count = pd.DataFrame(index=test_df.index)
signal_columns = [c for c in test_df.columns if c.endswith('_Signal')]

for sig_col in signal_columns:
    model_name = sig_col.rsplit('_', 1)[0]
    signals_to_count[model_name] = test_df[sig_col]

# 4. Evaluation ausführen
evaluation_table = evaluate_strategies(backtesting_results, signals_to_count, backtesting_transaction_costs)

# 5. Anzeige und Persistierung
print("\n--- UMFASSENDE EVALUATION (DYNAMISCH) ---")
display(evaluation_table)

evaluation_table.to_markdown(cfg.asset_path("evaluation_table"), index=True)
print(f"\nEvaluationstabelle erfolgreich unter {cfg.asset_path('evaluation_table')} persistiert.")

#--- Wir erhalten in diesem Schritt neben df und test_df sowie backtesting_results trades_df mit binären Handelssignalen ---

In [ ]:
# --- 06_MCS: Block-Bootstrap Robustness-Check ---

import matplotlib.pyplot as plt
import os

# Parameter aus zentraler Config
N_SIMULATIONS = cfg.evaluation.mcs.n_paths
BLOCK_SIZE = cfg.evaluation.mcs.block_length
RANDOM_SEED = cfg.evaluation.mcs.random_seed
SIM_YEARS = cfg.evaluation.mcs.sim_years
TRADING_DAYS = cfg.evaluation.mcs.trading_days_per_year
TOTAL_DAYS = SIM_YEARS * TRADING_DAYS

# Reproduzierbarkeit sicherstellen
np.random.seed(RANDOM_SEED)

# Definition der Szenarien aus zentraler Config
scenarios = {}
for scenario_name, scenario_cfg in vars(cfg.backtesting.sorr.scenarios).items():
    scenarios[scenario_name] = {
        "start": scenario_cfg.initial_capital,
        "withdrawal": scenario_cfg.initial_capital * scenario_cfg.annual_withdrawal_rate / 12,
        "fee": scenario_cfg.liquidity_fee
    }

# Tägliche Netto-Renditen der Strategien berechnen
daily_rets = backtesting_results.pct_change().dropna()

# Prüfen, ob Renditen vorhanden sind
if daily_rets.empty:
    raise ValueError("daily_rets ist leer. Prüfe die Datenquelle backtesting_results.")

# --- DYNAMISIERUNG: Automatische Zuordnung von Strategie zu Signal ---
def find_matching_signal_col(strategy_name, test_df_columns):
    if strategy_name == 'Buy_Hold':
        return None
    if f"{strategy_name}_Signal" in test_df_columns:
        return f"{strategy_name}_Signal"
    root_name = strategy_name.split('_')[0]
    potential_cols = [c for c in test_df_columns if root_name in c and "Signal" in c]
    if len(potential_cols) == 1:
        return potential_cols[0]
    for c in potential_cols:
        if strategy_name[:5] in c:
            return c
    return None

# Simulations-Loop
all_mc_summaries = []
mcs_paths_collector = {}

for sc_name, params in scenarios.items():
    print(f"Starte Monte Carlo Simulation für Szenario: {sc_name}")
    mc_results_scenario = {}
    
    for strategy in daily_rets.columns:
        final_capitals = []
        sig_col = find_matching_signal_col(strategy, test_df.columns)
        
        for s in range(N_SIMULATIONS):
            # --- Paired Block Bootstrap ---
            sim_rets = []
            sim_sigs = []
            
            while len(sim_rets) < TOTAL_DAYS:
                start_idx = np.random.randint(0, len(daily_rets) - BLOCK_SIZE)
                sim_rets.extend(daily_rets[strategy].iloc[start_idx : start_idx + BLOCK_SIZE].values)
                if sig_col:
                    sim_sigs.extend(test_df[sig_col].iloc[start_idx : start_idx + BLOCK_SIZE].values)
                else:
                    sim_sigs.extend([0] * BLOCK_SIZE)

            sim_rets = np.array(sim_rets[:TOTAL_DAYS])
            sim_sigs = np.array(sim_sigs[:TOTAL_DAYS])
            
            # --- Entnahme-Simulation ---
            cap = params["start"]
            capital_history = []
            
            for i in range(TOTAL_DAYS):
                cap *= (1 + sim_rets[i])
                
                if i % 21 == 0:
                    withdrawal_amt = params["withdrawal"]
                    if sim_sigs[i] == 0:
                        withdrawal_amt += (params["withdrawal"] * params["fee"])
                    cap -= withdrawal_amt
                    
                if cap <= 0:
                    cap = 0.0
                    capital_history.append(0.0)
                    remaining_days = TOTAL_DAYS - len(capital_history)
                    capital_history.extend([0.0] * remaining_days)
                    break
                else:
                    capital_history.append(cap)
            
            final_capitals.append(cap)
            
            path_id = f"{sc_name}_{strategy}_path_{s:03d}"
            mcs_paths_collector[path_id] = capital_history
        
        mc_results_scenario[strategy] = final_capitals
        
        if len(final_capitals) > 0:
            ruin_prob = np.mean(np.array(final_capitals) <= 0)
            median_wealth = np.median(final_capitals)
            
            all_mc_summaries.append({
                'Szenario': sc_name,
                'Strategie': strategy.replace('_', ' '),
                'Ruin-Wahrscheinlichkeit': f"{ruin_prob:.2%}",
                'Median Endkapital': f"{median_wealth:,.2f} €"
            })

    # Visualisierung pro Szenario
    if mc_results_scenario:
        plt.figure(figsize=(12, 6))
        plt.boxplot(mc_results_scenario.values(), 
                    tick_labels=[s.replace('_', ' ') for s in mc_results_scenario.keys()])
        plt.title(f"MCS {sc_name}: Verteilung des Endkapitals (Start: {params['start']:,.0f}€)")
        plt.ylabel(f"Endkapital nach {SIM_YEARS} Jahren in €")
        plt.axhline(y=0, color='red', linestyle='--')
        plt.grid(axis='y', alpha=0.3)
        plt.savefig(cfg.asset_path(f"mcs_boxplot_{sc_name.lower()}"), dpi=300, bbox_inches='tight')
        plt.show()

# Ergebnisse zusammenführen und als Markdown-Tabelle speichern
if all_mc_summaries:
    mc_multi_summary = pd.DataFrame(all_mc_summaries).set_index(['Szenario', 'Strategie'])
    mc_multi_summary.to_markdown(cfg.asset_path("mcs_summary"))
    display(mc_multi_summary)

print("Alle Monte Carlo Simulationen abgeschlossen.")

mcs_results = pd.DataFrame(mcs_paths_collector)
print(mcs_results)

#---

# Liste der Szenarien und Strategien dynamisch aus Config extrahieren
scenarios_list = list(vars(cfg.backtesting.sorr.scenarios).keys())
strategies = backtesting_results.columns

color_map = cfg.color_map
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# --- Plot: MCS Pfad-Verläufe ---
fig, axes = plt.subplots(len(scenarios_list), 1, figsize=(15, 6 * len(scenarios_list)), sharex=True)

for ax, sc_name in zip(axes, scenarios_list):
    print(f"Plotte Pfad-Verläufe für {sc_name}...")
    
    for strat in strategies:
        prefix = f"{sc_name}_{strat}_path_"
        strat_paths = mcs_results.filter(like=prefix)
        
        sample_paths = strat_paths
        
        color = color_map.get(strat, 'black')
        alpha = 0.05
        
        ax.plot(sample_paths, color=color, alpha=alpha, linewidth=1)
        ax.plot([], [], color=color, label=strat.replace('_', ' '))

    ax.set_title(f"MCS Pfad-Verläufe: Szenario {sc_name}")
    ax.set_ylabel("Kapital in €")
    ax.axhline(y=0, color='black', linewidth=1.5)
    ax.grid(alpha=0.2)
    ax.legend(loc='upper left', ncol=2)

plt.xlabel("Handelstage (T+N)")
plt.tight_layout()
plt.savefig(cfg.asset_path("mcs_paths"), dpi=300, bbox_inches='tight')
plt.show()

# --- Plot: MCS Quantils-Verteilung ---
fig, axes = plt.subplots(len(scenarios_list), 1, figsize=(15, 6 * len(scenarios_list)), sharex=True)

for ax, sc_name in zip(axes, scenarios_list):
    print(f"Berechne Quantile für {sc_name}...")
    
    for strat in strategies:
        prefix = f"{sc_name}_{strat}_path_"
        strat_paths = mcs_results.filter(like=prefix)
        
        upper_95 = strat_paths.quantile(0.95, axis=1)
        lower_05 = strat_paths.quantile(0.05, axis=1)
        median_50 = strat_paths.median(axis=1)
        
        color = color_map.get(strat, 'black')
        
        ax.fill_between(range(TOTAL_DAYS), lower_05, upper_95, color=color, alpha=0.15)
        ax.plot(median_50, color=color, linewidth=1.5, label=f"{strat.replace('_', ' ')} (Median)")

    ax.set_title(f"MCS Konfidenz-Intervalle (5% - 95%): Szenario {sc_name}")
    ax.set_ylabel("Kapital in €")
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1, label="Erschöpfungsgrenze")
    ax.grid(alpha=0.2)
    ax.legend(loc='upper left', ncol=2)

plt.xlabel("Handelstage (T+N)")
plt.tight_layout()
plt.savefig(cfg.asset_path("mcs_quantiles"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
output_path = cfg.data_path('mcs_data')

# Speichern als Parquet
mcs_results.to_parquet(output_path)

print(f"MCS-Dataframe erfolgreich unter {output_path} gespeichert.")